**Run Experiments 1 and 2 on the Development Split**

This notebook runs Experiments 1 and 2 of the experiment design (`./docs/experiment_design.pdf`, §4) on the development split, as the manifest (`./docs/MANIFEST.md`, §5–6) specifies them, and prepares the inputs of Experiment 3. Experiment 1 compares the candidate generators and Experiment 2 the equivalence dictionaries and where they act; each carries its strongest configurations forward. Every configuration is evaluated by five-fold cross-validation: the queries of each fold are answered from an index of every development person, with dictionaries learned from the other four folds, and each fold's parameters and display cut-off are tuned on the other four folds' queries. It reads `./eval/persons_dev.csv`, `./eval/queries_dev.csv`, `./eval/d1_forms.csv` and the signals of `./cache/signals`, asserts that no test identifier appears in them, and never opens a test file. `step_6_rank_and_freeze.ipynb` turns its runs into metrics, runs Experiment 3 and freezes the instrument. It outputs:

1. `./cache/runs/<config_id>.npz`: each configuration's answer to every query at its fold's tuned parameters (the 50 best persons with their scores, the display cut-off and the matching time), and the given-name and surname ranks of QS3–QS4
2. `./cache/development_runs.json`: each configuration's specification, tuned parameters, primary score and notes, and the configurations carried forward
3. `./cache/exp3_inputs.npz`: for every query, the token similarities of the three components carried into Experiment 3 and of the raw signals that the learned scorer K4 takes as features
4. `./review/link_sample.csv`: 100 dictionary links made by embedding similarity, left for human review

**1. Configuration and Data**

This cell defines the experiment's settings, reads the development files and the signals, and builds the index every configuration searches. It asserts that every person, query and relevant person is a development one, which keeps the test split sealed.

1. `persons` and `queries`: the development persons and queries, a query carrying its fold, target, relevant persons and the index tokens it masks
2. the token arrays: every name token of every person with its person, field (given or surname) and normalised form, sorted so that each person's tokens of one field are contiguous
3. `forms`: the vocabulary of normalised forms, the archive's forms first and then the curated dictionary's forms absent from the archive

In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# Configuration and data  —  the development index, queries and signals
#
# Similarities are computed between a query token and the vocabulary's
# forms, then copied to every token of each form. A person's given-name
# (surname) score for a query token is the best similarity among the
# person's given-name (surname) tokens, taken with one `maximum.reduceat`
# over the contiguous token segments of each person and field.
# ─────────────────────────────────────────────────────────────────────────────
import csv
import json
import time
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np

# ═════════════════════════════════════════════════════════════════════════════
# GLOBAL CONFIG
# ═════════════════════════════════════════════════════════════════════════════
EVAL_DIR       = Path("eval")                     # development inputs
SIGNALS_DIR    = Path("cache/signals")            # from step_4_compute_name_signals
RESULTS_DIR    = Path("results")                  # preflight.json
RUNS_DIR       = Path("cache/runs")               # one file per configuration
REVIEW_DIR     = Path("review")                   # human-review files
SEED           = 333               # review sampling
QUERY_SETS     = ("QS1", "QS2", "QS3", "QS4", "QS5")
TOP_K          = 50                # persons kept per query
TOKEN_CAP      = 50                # embedding matches kept per query token
CUTOFFS        = (0.70, 0.75, 0.80, 0.85, 0.90, 0.95)   # embedding cut-off c
TAU_LINKS      = (0.80, 0.85, 0.90)                     # ingestion-time link threshold
LAMBDA_EXP2    = 0.9               # dictionary penalty in Experiment 2
P3_TOP_K       = 5                 # P3's fixed-k retrieval
MAX_UNRELATED  = 3.0               # display cut-off constraint on unrelated@10
TAU_STEPS      = 41                # display cut-offs tried, from 0 to the top score
EMBEDDINGS     = ("E1", "E2", "E3a", "E3b", "E4")
LEXICAL        = ("L1", "L2", "L3", "L4")
RULE           = "─" * 72          # frames every printed report


def read_csv(path: Path) -> list[dict]:
    """Read the CSV file `path` into a list of dictionaries."""
    with path.open(encoding="utf-8", newline="") as file:
        return list(csv.DictReader(file))


def write_csv(path: Path, rows: list[dict]) -> None:
    """Write `rows` (dictionaries with the same keys) to the CSV file `path`."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0]), lineterminator="\n")
        writer.writeheader()
        writer.writerows(rows)


@dataclass
class Query:
    """One development query, ready to run against the index."""

    query_id: str
    query_set: str
    fold: int
    tokens: list[dict]
    target: int
    relevant: set[int]
    masked: np.ndarray        # indices of the index tokens hidden from this query


preflight = json.loads((RESULTS_DIR / "preflight.json").read_text(encoding="utf-8"))
persons = sorted(read_csv(EVAL_DIR / "persons_dev.csv"), key=lambda row: row["person_id"])
person_ids = [row["person_id"] for row in persons]
person_index = {pid: i for i, pid in enumerate(person_ids)}
person_fold = np.array([int(row["fold"]) for row in persons])
assert all(row["split"] == "dev" for row in persons), "a test person in persons_dev.csv"

# ── Vocabulary: archive forms first, then dictionary forms not in the archive ──
archive_forms = read_csv(SIGNALS_DIR / "index_forms.csv")
d1_forms = read_csv(EVAL_DIR / "d1_forms.csv")
forms = [(f["script"], f["norm"], f["display"]) for f in archive_forms]
form_index = {(script, norm): i for i, (script, norm, _) in enumerate(forms)}
for _form in d1_forms:
    if (_form["script"], _form["norm"]) not in form_index:
        form_index[(_form["script"], _form["norm"])] = len(forms)
        forms.append((_form["script"], _form["norm"], _form["display"]))
N_ARCHIVE = len(archive_forms)

# ── Index tokens, sorted by field then person so that segments are contiguous ──
_tokens = sorted(((t["field"] == "surname", person_index[row["person_id"]],
                   form_index[(t["script"], t["norm"])], t["column"])
                  for row in persons for t in json.loads(row["tokens"])))
tok_surname = np.array([t[0] for t in _tokens])
tok_person = np.array([t[1] for t in _tokens])
tok_form = np.array([t[2] for t in _tokens])
tok_column = [t[3] for t in _tokens]
seg_start = np.flatnonzero(np.r_[True, (np.diff(tok_person) != 0) | (np.diff(tok_surname) != 0)])
seg_person, seg_surname = tok_person[seg_start], tok_surname[seg_start]
form_token_count = np.bincount(tok_form, minlength=N_ARCHIVE)


def make_query(row: dict) -> Query:
    """Build a `Query` from one row of `queries_dev.csv`."""
    (target,) = [person_index[pid] for pid in json.loads(row["target_person_ids"])]
    columns = set(filter(None, row["masked_fields"].split(";")))
    masked = np.flatnonzero((tok_person == target)
                            & np.isin(np.array(tok_column), sorted(columns)))
    return Query(row["query_id"], row["set"], int(row["fold"]), json.loads(row["query_tokens"]),
                 target, {person_index[pid] for pid in json.loads(row["relevant_person_ids"])},
                 masked)


queries = [make_query(row) for row in read_csv(EVAL_DIR / "queries_dev.csv")]
query_set_index = np.array([QUERY_SETS.index(q.query_set) for q in queries])
query_fold = np.array([q.fold for q in queries])
query_target = np.array([q.target for q in queries])

# ── Signals of step_4_compute_name_signals ─────────────────────────────────────────
dm_codes = {k: set(v) for k, v in
            json.loads((SIGNALS_DIR / "dm_codes.json").read_text(encoding="utf-8")).items()}
bmpm_codes = json.loads((SIGNALS_DIR / "bmpm_codes.json").read_text(encoding="utf-8"))
_trigram = np.load(SIGNALS_DIR / "trigram.npz")
trigram_rows = defaultdict(lambda: (np.zeros(0, int), np.zeros(0)))
for _i, _key in enumerate(_trigram["query_keys"]):
    _mask = _trigram["rows"] == _i
    trigram_rows[str(_key)] = (_trigram["cols"][_mask], _trigram["values"][_mask])
encode_latency = defaultdict(dict)
for _row in read_csv(SIGNALS_DIR / "query_latency.csv"):
    encode_latency[_row["generator"]][_row["query_id"]] = float(_row["ms"])
print(f"✓ Configuration and data loaded ({len(persons)} persons, {len(_tokens)} tokens, "
      f"{N_ARCHIVE} archive forms, {len(forms) - N_ARCHIVE} dictionary-only forms, "
      f"{len(queries)} queries).")

✓ Configuration and data loaded (507 persons, 1844 tokens, 995 archive forms, 436 dictionary-only forms, 1743 queries).


**2. Generators**

This cell turns each signal into the similarity of a query token to every archive form, for the eight generators of Experiment 1 and the initial-sound rule D4:

1. **E1–E4**: the cosine of the token's embedding to each form's embedding. E4 has one model per fold, the one fine-tuned without that fold
2. **L1**: 1 for the identical normalised form; **L2**: the `pg_trgm` similarity, within the token's script; **L3**: 1 for a form sharing a Daitch–Mokotoff code (Hebrew romanised first); **L4**: 1 for a form sharing a Beider–Morse token in the approximate or the Hebrew-value pass
3. **D4**: 1 for a form whose Daitch–Mokotoff code starts with the same digit as one of the token's codes, used only as a K4 feature

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# Generators  —  query token → similarity to every archive form
#
# Embedding vectors were computed in bulk by step_4_compute_name_signals, whose
# timing of each model's query encoding enters the reported latency; the
# matching done here (a matrix–vector product, or a code look-up through
# an inverted index) is timed by the evaluation loop.
# ─────────────────────────────────────────────────────────────────────────────
if "queries" not in globals():
    raise RuntimeError("Run the 'Configuration and Data' cell first.")


def load_embeddings(name: str) -> tuple[dict[str, np.ndarray], np.ndarray]:
    """Return a text → vector map and the vocabulary's vectors for one model file."""
    stored = np.load(SIGNALS_DIR / f"emb_{name}.npz")
    by_text = dict(zip(map(str, stored["texts"]), stored["vectors"]))
    return by_text, np.stack([by_text[display] for _, _, display in forms])


def code_index(codes_of) -> dict[str, np.ndarray]:
    """Return an inverted index code → archive forms, given `codes_of(form key)`."""
    index = defaultdict(list)
    for i, (script, norm, _) in enumerate(forms[:N_ARCHIVE]):
        for code in codes_of(f"{script}|{norm}"):
            index[code].append(i)
    return {code: np.array(members) for code, members in index.items()}


def first_digits(key: str) -> set[str]:
    """Return the first digits of a form's Daitch–Mokotoff codes (the D4 key)."""
    return {code[0] for code in dm_codes.get(key, ())}


embeddings = {}
for _name in [*EMBEDDINGS[:4], *[f"E4_fold{k}" for k in range(1, 6)]]:
    _model = _name.split("_")[0]
    if preflight[_model]["status"] == "ok":
        embeddings[_name] = load_embeddings(_name)
CODE_INDEXES = {
    "L3": code_index(lambda key: dm_codes.get(key, ())),
    "L4a": code_index(lambda key: bmpm_codes.get(key, {}).get("approx", ())),
    "L4h": code_index(lambda key: bmpm_codes.get(key, {}).get("hebrew", ())),
    "D4": code_index(first_digits),
}


def embedding_name(generator: str, fold: int) -> str:
    """Return the embedding file behind `generator` for queries of `fold`."""
    return f"E4_fold{fold}" if generator == "E4" else generator


def form_cosines(generator: str, fold: int, text: str, rows=slice(None)) -> np.ndarray:
    """Return the cosine of `text` to the vocabulary forms `rows` (clipped at 0)."""
    by_text, vectors = embeddings[embedding_name(generator, fold)]
    return np.clip(vectors[rows] @ by_text[text], 0, None)


def shared_code_forms(index_name: str, codes) -> np.ndarray:
    """Return the archive forms sharing at least one of `codes` in an inverted index."""
    members = [CODE_INDEXES[index_name][c] for c in codes if c in CODE_INDEXES[index_name]]
    return np.unique(np.concatenate(members)) if members else np.zeros(0, int)


def lexical_similarity(generator: str, token: dict) -> np.ndarray:
    """Return a lexical generator's similarity of `token` to every archive form."""
    key, sims = f"{token['script']}|{token['norm']}", np.zeros(N_ARCHIVE)
    if generator == "L1":
        exact = form_index.get((token["script"], token["norm"]), N_ARCHIVE)
        sims[exact if exact < N_ARCHIVE else []] = 1.0
    elif generator == "L2":
        columns, values = trigram_rows[key]
        sims[columns] = values
    elif generator in ("L3", "D4"):
        codes = dm_codes.get(key, set()) if generator == "L3" else first_digits(key)
        sims[shared_code_forms(generator, codes)] = 1.0
    elif generator == "L4":
        for index_name, mode in (("L4a", "approx"), ("L4h", "hebrew")):
            sims[shared_code_forms(index_name, bmpm_codes.get(key, {}).get(mode, ()))] = 1.0
    return sims


def generator_similarity(generator: str, fold: int, token: dict) -> np.ndarray:
    """Return any generator's similarity of `token` to every archive form."""
    if generator in EMBEDDINGS:
        return form_cosines(generator, fold, token["display"], slice(0, N_ARCHIVE))
    return lexical_similarity(generator, token)


print(f"{RULE}\nGenerators\n{RULE}")
for _generator in [*EMBEDDINGS, *LEXICAL]:
    _ready = (_generator in LEXICAL or embedding_name(_generator, 1) in embeddings)
    print(f"  {'OK ' if _ready else '!! '}{_generator:<4} "
          + ("ready" if _ready else preflight[_generator]["status"]))
print(f"{RULE}\n  inverted indexes: " + ", ".join(f"{name} {len(index)} codes"
                                             for name, index in CODE_INDEXES.items()))

────────────────────────────────────────────────────────────────────────
Generators
────────────────────────────────────────────────────────────────────────
  OK E1   ready
  OK E2   ready
  OK E3a  ready
  OK E3b  ready
  OK E4   ready
  OK L1   ready
  OK L2   ready
  OK L3   ready
  OK L4   ready
────────────────────────────────────────────────────────────────────────
  inverted indexes: L3 600 codes, L4a 5996 codes, L4h 1397 codes, D4 10 codes


**3. Dictionaries and Placements**

This cell builds the equivalence dictionaries and the three ways Experiment 2 lets a dictionary act. A dictionary is a set of groups of equivalent forms:

1. **D1**: the curated given-name groups; **D2**: for each fold, the Spanish–Hebrew pairs of given-name tokens and of surname tokens recorded on the same person in the four other folds, kept at a support of at least two persons and merged into groups by connected components; **D3**: D1 and D2 together, groups sharing a form merged
2. **Ingestion-time links**: every archive form joins the groups of its own form, or else those of its nearest dictionary form by the generator's cosine if that cosine reaches `τ_link`
3. **P1**, dictionary only: an exact look-up of the query token, returning the persons linked to its groups (and its exact matches); **P2**, dictionary first: the query token and every form of its groups are embedded and compared with the archive; **P3**, embedding first: the token's neighbours in a search vocabulary (the archive's forms, or those and the dictionary's) bring their groups, whose linked persons are returned

A dictionary-derived match is returned with its seed similarity, which the ranker multiplies by the penalty λ.

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Dictionaries and placements  —  D1–D3, links, and P1–P3
#
# A placement returns two arrays over the archive forms: `direct`, the
# similarity of a direct match, and `derived`, the seed similarity of a
# dictionary-derived match. Groups are keyed by name; the forms in them
# by vocabulary index, so that a dictionary form outside the archive can
# still seed a group.
# ─────────────────────────────────────────────────────────────────────────────
if "generator_similarity" not in globals():
    raise RuntimeError("Run the 'Generators' cell first.")


@dataclass
class Dictionary:
    """Groups of equivalent forms: `groups` maps a group to its forms, and
    `groups_of` maps a form (vocabulary index) to its groups."""

    name: str
    groups: dict[str, set[int]]
    groups_of: dict[int, set[str]]


def make_dictionary(name: str, groups: dict[str, set[int]]) -> Dictionary:
    """Return a `Dictionary` with the reverse index of `groups`."""
    groups_of = defaultdict(set)
    for group, members in groups.items():
        for form in members:
            groups_of[form].add(group)
    return Dictionary(name, groups, dict(groups_of))


def merge_groups(name: str, groups: dict[str, set[int]]) -> Dictionary:
    """Merge groups that share a form (connected components) into one dictionary."""
    merged, owner = [], {}
    for members in groups.values():
        overlapping = {owner[f] for f in members if f in owner}
        union = set(members).union(*(merged[i] for i in overlapping))
        for i in overlapping:
            merged[i] = set()
        merged.append(union)
        for form in union:
            owner[form] = len(merged) - 1
    return make_dictionary(name, {f"{name}_{i}": m for i, m in enumerate(merged) if m})


def d2_groups(excluded_fold: int) -> dict[str, set[int]]:
    """Return D2's pair groups learned from the persons outside `excluded_fold`."""
    support = Counter()
    for row in persons:
        if int(row["fold"]) == excluded_fold:
            continue
        tokens = json.loads(row["tokens"])
        for field_name in ("given", "surname"):
            in_field = [t for t in tokens if t["field"] == field_name]
            pick = {s: {form_index[(t["script"], t["norm"])] for t in in_field if t["script"] == s}
                    for s in ("latin", "hebrew")}
            support.update((a, b) for a in pick["latin"] for b in pick["hebrew"])
    return {f"pair_{a}_{b}": {a, b} for (a, b), count in support.items() if count >= 2}


D1_GROUPS = defaultdict(set)
for _form in d1_forms:
    D1_GROUPS[f"d1_{_form['group_id']}"].add(form_index[(_form["script"], _form["norm"])])
dictionaries = {"D1": {k: make_dictionary("D1", D1_GROUPS) for k in range(1, 6)},
                "D2": {k: merge_groups("D2", d2_groups(k)) for k in range(1, 6)}}
dictionaries["D3"] = {k: merge_groups("D3", {**D1_GROUPS, **dictionaries["D2"][k].groups})
                      for k in range(1, 6)}
_LINK_CACHE = {}
NO_FORMS = np.zeros(0, dtype=int)


@dataclass
class Links:
    """Ingestion-time links of the archive forms to one dictionary's groups.

    Attributes:
        members: each group's linked archive forms.
        groups_of_form: each archive form's linked groups.
        non_exact: the forms linked through a nearest dictionary form, with
            that form (vocabulary index) and the cosine to it.
    """

    members: dict[str, np.ndarray]
    groups_of_form: list[set[str]]
    non_exact: dict[int, tuple[int, float]]


def links(dictionary: Dictionary, generator: str, fold: int, tau_link) -> Links:
    """Link every archive form to dictionary groups: its own form's groups, else
    those of its nearest dictionary form if the cosine reaches `tau_link` (cached).
    Without a generator (P1) only exact links are made."""
    key = (dictionary.name, fold, generator, tau_link)
    if key not in _LINK_CACHE:
        members, groups_of_form, non_exact = defaultdict(list), [], {}
        dict_forms = np.array(sorted(dictionary.groups_of))
        cosines = (form_cosines_matrix(generator, fold, dict_forms)
                   if generator and tau_link else None)
        for form in range(N_ARCHIVE):
            groups = dictionary.groups_of.get(form, set())
            if not groups and cosines is not None:
                best = int(np.argmax(cosines[form]))
                if cosines[form, best] >= tau_link:
                    groups = dictionary.groups_of[dict_forms[best]]
                    non_exact[form] = (int(dict_forms[best]), float(cosines[form, best]))
            groups_of_form.append(groups)
            for group in groups:
                members[group].append(form)
        _LINK_CACHE[key] = Links({g: np.array(f) for g, f in members.items()},
                                 groups_of_form, non_exact)
    return _LINK_CACHE[key]


def form_cosines_matrix(generator: str, fold: int, columns: np.ndarray) -> np.ndarray:
    """Return the cosines between every archive form and the vocabulary forms `columns`."""
    _, vectors = embeddings[embedding_name(generator, fold)]
    return vectors[:N_ARCHIVE] @ vectors[columns].T


@dataclass
class Lookup:
    """What every placement needs to know about one query token: the token, the
    query's fold, that fold's dictionary, and the token's own vocabulary form
    (None if new) with that form's groups."""

    token: dict
    fold: int
    dictionary: Dictionary
    own: int | None
    own_groups: set[str]


def place_p1(lookup: Lookup):
    """P1, dictionary only: the token's exact matches, and every archive form
    linked (exactly) to the token's own groups, with seed similarity 1."""
    derived = np.zeros(N_ARCHIVE)
    linked = links(lookup.dictionary, "", lookup.fold, None)
    for group in lookup.own_groups:
        derived[linked.members.get(group, NO_FORMS)] = 1.0
    return lexical_similarity("L1", lookup.token), derived


def place_p2(generator: str, theta: dict, lookup: Lookup):
    """P2, dictionary first: the token and each form of its groups are compared
    with the archive; a form's matches (cosine ≥ c) are dictionary-derived."""
    direct = generator_similarity(generator, lookup.fold, lookup.token)
    direct[direct < theta["c"]] = 0
    derived = np.zeros(N_ARCHIVE)
    expansions = set().union(*(lookup.dictionary.groups[g] for g in lookup.own_groups))
    for form in expansions - {lookup.own}:
        cosines = form_cosines(generator, lookup.fold, forms[form][2], slice(0, N_ARCHIVE))
        derived = np.maximum(derived, np.where(cosines >= theta["c"], cosines, 0))
    return direct, derived


def p3_neighbours(config, theta: dict, lookup: Lookup, available: np.ndarray):
    """Return P3's neighbours of the token in the search vocabulary (archive forms
    with an unmasked token, plus the dictionary's forms if so configured): their
    vocabulary indices and cosines, by cut-off `c` (at most 50) or the five nearest."""
    vocabulary = np.flatnonzero(np.r_[available, np.zeros(len(forms) - N_ARCHIVE, bool)])
    if config.vocab == "archive+dict":
        vocabulary = np.union1d(vocabulary, sorted(lookup.dictionary.groups_of))
    cosines = form_cosines(config.generator, lookup.fold, lookup.token["display"], vocabulary)
    order = np.argsort(-cosines, kind="stable")
    keep = (order[:P3_TOP_K] if config.retrieval == "top5"
            else order[cosines[order] >= theta["c"]][:TOKEN_CAP])
    return vocabulary[keep], cosines[keep]


def place_p3(config, theta: dict, lookup: Lookup, available: np.ndarray):
    """P3, embedding first: the token's neighbours are direct matches (if archive
    forms) and seed their groups, whose linked archive forms are
    dictionary-derived matches at the seed's cosine."""
    linked = links(lookup.dictionary, config.generator, lookup.fold, theta["tau_link"])
    direct, derived = np.zeros(N_ARCHIVE), np.zeros(N_ARCHIVE)
    seeds = [(1.0, lookup.own_groups)] if lookup.own_groups else []
    for form, cosine in zip(*p3_neighbours(config, theta, lookup, available)):
        if form < N_ARCHIVE:
            direct[form] = cosine
            seeds.append((float(cosine), linked.groups_of_form[form]))
        else:
            seeds.append((float(cosine), lookup.dictionary.groups_of.get(form, set())))
    for similarity, groups in seeds:
        for group in groups:
            members = linked.members.get(group, NO_FORMS)
            derived[members] = np.maximum(derived[members], similarity)
    return direct, derived


def place_dictionary(config, theta: dict, fold: int, token: dict, available: np.ndarray):
    """Return the (direct, derived) similarities of `token` under P1, P2 or P3.

    Args:
        config: the configuration (generator, dictionary, placement, vocab, retrieval).
        theta: the tuned parameters `c` and `tau_link`.
        fold: the fold of the query, which selects D2, E4 and the links.
        token: the query token.
        available: archive forms with at least one token not masked.
    """
    dictionary = dictionaries[config.dictionary][fold]
    own = form_index.get((token["script"], token["norm"]))
    lookup = Lookup(token, fold, dictionary, own,
                    dictionary.groups_of.get(own, set()) if own is not None else set())
    if config.placement == "P1":
        return place_p1(lookup)
    if config.placement == "P2":
        return place_p2(config.generator, theta, lookup)
    return place_p3(config, theta, lookup, available)


print(f"{RULE}\nDictionaries (per fold: groups, forms)\n{RULE}")
for _name, _by_fold in dictionaries.items():
    print(f"  {_name}  " + "   ".join(f"fold {k}: {len(d.groups):>3} groups"
                                     for k, d in _by_fold.items()))
print(f"{RULE}\n  D1 holds {len(D1_GROUPS)} curated groups; D2 is rebuilt for every fold")

────────────────────────────────────────────────────────────────────────
Dictionaries (per fold: groups, forms)
────────────────────────────────────────────────────────────────────────
  D1  fold 1: 105 groups   fold 2: 105 groups   fold 3: 105 groups   fold 4: 105 groups   fold 5: 105 groups
  D2  fold 1:  42 groups   fold 2:  41 groups   fold 3:  46 groups   fold 4:  44 groups   fold 5:  40 groups
  D3  fold 1: 125 groups   fold 2: 125 groups   fold 3: 130 groups   fold 4: 126 groups   fold 5: 125 groups
────────────────────────────────────────────────────────────────────────
  D1 holds 105 curated groups; D2 is rebuilt for every fold


**4. Scoring and Tuning**

This cell defines how a configuration answers a query and how its parameters are tuned, shared by both experiments:

1. **Person scoring** (manifest §5.2): for each query token, a person's given-name score is its best similarity among its given-name tokens, and likewise for surnames. A one-token query scores `max(g, s)`; a longer query scores the best mean of `g_i` and `s_j` over pairs of different tokens, so that a full-name query ranks a person highly only when both parts match
2. **Evaluation**: a configuration answers every development query once per point of its parameter grid (cut-off `c`, link threshold `τ_link`), keeping the 50 best persons. Each fold's parameters and display cut-off `τ_d` are chosen on the other four folds' queries: the largest macro-average R@10 over QS1–QS5 with at most `MAX_UNRELATED` unrelated persons among the first ten shown. One set chosen on all five folds together is what a frozen configuration takes to the test split
3. **Subsets**: for a QS3 or QS4 query, the rank of the target's other-script given-name (surname) form among the archive forms most similar to the query's given-name (surname) token, which isolates each half of the cross-script match
4. `evaluate` stores the result in `evaluations` and writes `./cache/runs/<config_id>.npz`

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# Scoring and tuning  —  the engine both experiments share
#
# A run holds, for every query, the ids and scores of its 50 best persons
# and the milliseconds the matching took. The display cut-off only hides
# the tail of a ranked list, so every τ_d is scored from the same run.
# ─────────────────────────────────────────────────────────────────────────────
if "place_dictionary" not in globals():
    raise RuntimeError("Run the 'Dictionaries and Placements' cell first.")


@dataclass(frozen=True)
class Config:
    """One search configuration of Experiment 1 or 2. Its ranker is K1 without a
    dictionary and K2 (λ = 0.9) with one, and its aggregation is the mean."""

    config_id: str
    experiment: str
    generator: str = ""
    dictionary: str = "D0"
    placement: str = ""
    vocab: str = ""
    retrieval: str = ""

    def spec(self) -> dict:
        """Return the configuration's settings, ranker and aggregation included."""
        return asdict(self) | {"ranker": "K2" if self.placement else "K1",
                               "aggregation": "mean"}


def theta_grid(config: Config) -> list[dict]:
    """Return the parameter points a configuration is evaluated at."""
    embedding = config.generator in EMBEDDINGS
    uses_cutoff = embedding and (config.placement in ("", "P2") or config.retrieval == "cutoff")
    return [{"c": c, "tau_link": t, "lam": LAMBDA_EXP2 if config.placement else 1.0}
            for c in (CUTOFFS if uses_cutoff else (None,))
            for t in (TAU_LINKS if config.placement == "P3" else (None,))]


def cap_matches(similarities: np.ndarray) -> np.ndarray:
    """Keep only the TOKEN_CAP largest similarities of one query token."""
    if np.count_nonzero(similarities) <= TOKEN_CAP:
        return similarities
    kept = np.zeros_like(similarities)
    best = np.argpartition(-similarities, TOKEN_CAP)[:TOKEN_CAP]
    kept[best] = similarities[best]
    return kept


def component_similarity(config: Config, theta: dict, query: Query):
    """Return, per query token, the direct and the dictionary-derived similarity
    of every index token, with the query's masked tokens set to zero."""
    available = form_token_count > np.bincount(tok_form[query.masked], minlength=N_ARCHIVE)
    direct_rows, derived_rows = [], []
    for token in query.tokens:
        if config.placement:
            direct, derived = place_dictionary(config, theta, query.fold, token, available)
        else:
            direct = generator_similarity(config.generator, query.fold, token)
            derived = np.zeros(N_ARCHIVE)
            if config.generator in EMBEDDINGS:
                direct[direct < theta["c"]] = 0
        direct, derived = direct[tok_form], derived[tok_form]
        direct[query.masked], derived[query.masked] = 0, 0
        if config.generator in EMBEDDINGS and config.placement != "P3":
            direct = cap_matches(direct)
        direct_rows.append(direct)
        derived_rows.append(derived)
    return np.array(direct_rows), np.array(derived_rows)


def field_scores(similarities: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Return per query token every person's given-name and surname scores."""
    maxima = np.maximum.reduceat(similarities, seg_start, axis=1)
    given, surname = (np.zeros((len(similarities), len(persons))) for _ in range(2))
    given[:, seg_person[~seg_surname]] = maxima[:, ~seg_surname]
    surname[:, seg_person[seg_surname]] = maxima[:, seg_surname]
    return given, surname


def mean_aggregate(given: np.ndarray, surname: np.ndarray) -> np.ndarray:
    """Return every person's score: max(g, s) for one token, else the best mean
    of g_i and s_j over pairs of different query tokens."""
    if len(given) == 1:
        return np.maximum(given[0], surname[0])
    pairs = (given[:, None, :] + surname[None, :, :]) / 2
    pairs[np.arange(len(given)), np.arange(len(given))] = 0
    return pairs.max(axis=(0, 1))


def top_persons(scores: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Return the TOP_K best persons with a score above zero (ties by person id),
    padded with -1 and 0."""
    candidates = np.flatnonzero(scores > 0)
    ranked = candidates[np.lexsort((candidates, -scores[candidates]))][:TOP_K]
    ids, values = np.full(TOP_K, -1), np.zeros(TOP_K)
    ids[:len(ranked)], values[:len(ranked)] = ranked, scores[ranked]
    return ids, values


def run_grid_point(config: Config, theta: dict) -> dict:
    """Answer every query at one parameter point; return the ids, scores and
    matching milliseconds of every query's 50 best persons."""
    run = {"ids": np.full((len(queries), TOP_K), -1), "scores": np.zeros((len(queries), TOP_K)),
           "ms": np.zeros(len(queries))}
    for i, query in enumerate(queries):
        started = time.perf_counter()
        direct, derived = component_similarity(config, theta, query)
        scores = mean_aggregate(*field_scores(np.maximum(direct, theta["lam"] * derived)))
        run["ids"][i], run["scores"][i] = top_persons(scores)
        run["ms"][i] = (time.perf_counter() - started) * 1000
    return run


RELEVANT = np.zeros((len(queries), len(persons)), dtype=bool)
for _i, _query in enumerate(queries):
    RELEVANT[_i, sorted(_query.relevant)] = True


def metrics_at(run: dict, taus: np.ndarray, rows: np.ndarray) -> dict[str, np.ndarray]:
    """Return per query (of `rows`) and per display cut-off the R@10, reciprocal
    rank and unrelated@10 of the run's displayed list."""
    ids, scores = run["ids"][rows], run["scores"][rows]
    relevant = np.take_along_axis(RELEVANT[rows], ids.clip(0), axis=1) & (ids >= 0)
    shown = ((scores[:, :, None] >= taus) & (scores[:, :, None] > 0)).sum(axis=1)
    shown10 = np.minimum(shown, 10)
    cumulative = np.concatenate([np.zeros((len(ids), 1), int), relevant.cumsum(axis=1)], axis=1)
    hits = np.take_along_axis(cumulative, shown10, axis=1)
    first = np.where(relevant.any(axis=1), relevant.argmax(axis=1) + 1, TOP_K + 1)[:, None]
    return {"r10": hits / np.minimum(10, RELEVANT[rows].sum(axis=1))[:, None],
            "rr": np.where(first <= shown, 1.0 / first, 0.0), "unrelated": shown10 - hits}


def macro(values: np.ndarray, rows: np.ndarray) -> np.ndarray:
    """Return the macro-average over query sets of `values` (rows × cut-offs)."""
    sets = query_set_index[rows]
    return np.mean([values[sets == s].mean(axis=0) for s in np.unique(sets)], axis=0)


def choose(runs: list[dict], taus: np.ndarray, rows: np.ndarray) -> tuple[int, float]:
    """Choose the grid point and display cut-off with the best macro R@10 on
    `rows`, subject to mean unrelated@10 ≤ MAX_UNRELATED (ties: MRR, then the
    earlier point and the lower cut-off). Without a feasible cut-off, the one
    with the fewest unrelated persons is taken."""
    best_key, best = None, (0, float(taus[-1]))
    for i, run in enumerate(runs):
        metrics = metrics_at(run, taus, rows)
        r10, mrr = macro(metrics["r10"], rows), macro(metrics["rr"], rows)
        unrelated = metrics["unrelated"].mean(axis=0)
        for j, tau in enumerate(taus):
            feasible = unrelated[j] <= MAX_UNRELATED
            key = (feasible, r10[j] if feasible else -unrelated[j], mrr[j], -i, -j)
            if best_key is None or key > best_key:
                best_key, best = key, (i, float(tau))
    return best


def subset_ranks(config: Config, theta: dict, query: Query) -> tuple[int, int]:
    """Return, for the given name and the surname of a QS3/QS4 query, the rank of
    the target's best-ranked unmasked form of that field among the archive forms
    most similar to the query's token(s) of that field (0 if not ranked, -1 if
    the query or the target lacks the field)."""
    direct, derived = component_similarity(config, theta, query)
    similarities = np.maximum(direct, theta["lam"] * derived)
    unmasked = np.setdiff1d(np.flatnonzero(tok_person == query.target), query.masked)
    ranks = []
    for field_name, surname in (("given", False), ("surname", True)):
        rows = [i for i, t in enumerate(query.tokens) if t["field"] == field_name]
        targets = set(tok_form[unmasked[tok_surname[unmasked] == surname]])
        if not rows or not targets:
            ranks.append(-1)
            continue
        form_similarity = np.zeros(N_ARCHIVE)
        np.maximum.at(form_similarity, tok_form, similarities[rows].max(axis=0))
        ranked = np.flatnonzero(form_similarity > 0)
        ranked = ranked[np.lexsort((ranked, -form_similarity[ranked]))]
        positions = [p + 1 for p, form in enumerate(ranked) if form in targets]
        ranks.append(positions[0] if positions else 0)
    return ranks[0], ranks[1]


evaluations = {}
FOLDS = (1, 2, 3, 4, 5)


def evaluate(config: Config) -> str:
    """Evaluate a configuration over its grid, tune it per fold and pooled,
    and save its run at each fold's choice to RUNS_DIR. Returns its id."""
    thetas = theta_grid(config)
    runs = [run_grid_point(config, theta) for theta in thetas]
    taus = np.linspace(0, max(float(run["scores"].max()) for run in runs) or 1, TAU_STEPS)
    choice = {f: choose(runs, taus, query_fold != f) for f in FOLDS}
    pooled = choose(runs, taus, np.ones(len(queries), bool))
    final = {key: np.zeros_like(runs[0][key]) for key in ("ids", "scores", "ms")}
    final["tau"], final["subsets"] = np.zeros(len(queries)), np.full((len(queries), 2), -1)
    for i, query in enumerate(queries):
        point, final["tau"][i] = choice[query.fold]
        for key in ("ids", "scores", "ms"):
            final[key][i] = runs[point][key][i]
        if query.query_set in ("QS3", "QS4"):
            final["subsets"][i] = subset_ranks(config, thetas[point], query)
    r10 = np.zeros(len(queries))
    for f in FOLDS:
        rows = query_fold == f
        r10[rows] = metrics_at(final, np.array([choice[f][1]]), rows)["r10"][:, 0]
    np.savez_compressed(RUNS_DIR / f"{config.config_id}.npz",
                        query_ids=np.array([q.query_id for q in queries]), **final)
    evaluations[config.config_id] = {
        "spec": config.spec(), "primary": float(np.mean(
            [macro(r10[query_fold == f][:, None], query_fold == f)[0] for f in FOLDS])),
        "fold_theta": {f: thetas[i] for f, (i, _) in choice.items()},
        "fold_tau": {f: tau for f, (_, tau) in choice.items()},
        "pooled_theta": thetas[pooled[0]], "pooled_tau": pooled[1], "notes": ""}
    return config.config_id


def report(title: str, ids: list[str]) -> None:
    """Print the configurations' primary scores, best first."""
    print(f"{RULE}\n{title}\n{RULE}")
    for config_id in sorted(ids, key=lambda c: -evaluations[c]["primary"]):
        e = evaluations[config_id]
        params = ", ".join(f"{k} {v}" for k, v in e["pooled_theta"].items() if v is not None)
        print(f"  {config_id:<36} R@10 {e['primary']:.3f}   ({params}, τ_d {e['pooled_tau']:.2f})")


RUNS_DIR.mkdir(parents=True, exist_ok=True)
print(f"✓ Scoring and tuning defined ({TAU_STEPS} display cut-offs, unrelated@10 ≤ "
      f"{MAX_UNRELATED:.0f}).")

✓ Scoring and tuning defined (41 display cut-offs, unrelated@10 ≤ 3).


**5. Experiment 1: Candidate Generators (RQ1)**

This cell evaluates the nine candidate generators with no dictionary (D0), ranking K1 and mean aggregation: the embedding models E1, E2, E3a, E3b and E4 (E1 fine-tuned per fold) and the lexical baselines L1–L4. Embedding generators keep the tokens with a cosine of at least `c`, at most 50 per query token, with `c` tuned per fold. A generator the preflight could not obtain is recorded as not run. It carries forward the two best embedding generators and the best lexical generator by primary score: the macro-average R@10 over QS1–QS5, averaged over the five folds.

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# Experiment 1  —  generators, no dictionary, K1, mean aggregation
# ─────────────────────────────────────────────────────────────────────────────
if "evaluate" not in globals():
    raise RuntimeError("Run the 'Scoring and Tuning' cell first.")

not_run_configs = {}
exp1_ids = []
for _generator in (*EMBEDDINGS, *LEXICAL):
    _config = Config(f"exp1_{_generator}", "exp1", generator=_generator)
    if _generator in LEXICAL or embedding_name(_generator, 1) in embeddings:
        exp1_ids.append(evaluate(_config))
    else:
        not_run_configs[_config.config_id] = (_config.spec(), preflight[_generator]["status"])
_ranked = sorted(exp1_ids, key=lambda c: -evaluations[c]["primary"])
carried_embeddings = [c for c in _ranked if evaluations[c]["spec"]["generator"] in EMBEDDINGS][:2]
best_lexical = next(c for c in _ranked if evaluations[c]["spec"]["generator"] in LEXICAL)
report("Experiment 1: generators (fold-mean macro R@10, pooled parameters)", exp1_ids)
print(f"{RULE}\n  carried forward: {', '.join(carried_embeddings)} and {best_lexical}")

────────────────────────────────────────────────────────────────────────
Experiment 1: generators (fold-mean macro R@10, pooled parameters)
────────────────────────────────────────────────────────────────────────
  exp1_E1                              R@10 0.741   (c 0.7, lam 1.0, τ_d 0.50)
  exp1_E4                              R@10 0.735   (c 0.7, lam 1.0, τ_d 0.50)
  exp1_E2                              R@10 0.711   (c 0.7, lam 1.0, τ_d 0.50)
  exp1_E3b                             R@10 0.602   (c 0.85, lam 1.0, τ_d 0.50)
  exp1_E3a                             R@10 0.568   (c 0.7, lam 1.0, τ_d 0.50)
  exp1_L3                              R@10 0.495   (lam 1.0, τ_d 0.53)
  exp1_L2                              R@10 0.446   (lam 1.0, τ_d 0.53)
  exp1_L4                              R@10 0.438   (lam 1.0, τ_d 0.53)
  exp1_L1                              R@10 0.085   (lam 1.0, τ_d 0.53)
────────────────────────────────────────────────────────────────────────
  carried forward: exp1_E1, ex

**6. Experiment 2: Dictionary Source and Placement (RQ2)**

This cell compares the dictionaries D1–D3 in each placement, with Experiment 1's carried embeddings, ranking K2 with λ = 0.9 and mean aggregation:

1. P1 with D1, D2 and D3, which uses no generator
2. P2 with each carried embedding and each dictionary
3. P3 with the best embedding and D3 under the four settings of search vocabulary (archive forms, or those and the dictionary's) and retrieval (cut-off `c`, or the five nearest), then P3 with each carried embedding and each dictionary at the best setting

It carries forward the two best configurations, notes each D2 configuration's link count and share of non-exact links, and writes 100 random non-exact links of the best P3 configuration to `./review/link_sample.csv` for review.

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# Experiment 2  —  dictionaries D1–D3 × placements P1–P3, K2 with λ = 0.9
# ─────────────────────────────────────────────────────────────────────────────
if "carried_embeddings" not in globals():
    raise RuntimeError("Run the 'Experiment 1' cell first.")

DICTIONARY_NAMES = ("D1", "D2", "D3")
CARRIED_GENERATORS = [evaluations[c]["spec"]["generator"] for c in carried_embeddings]


def p3_config(generator: str, dictionary: str, vocab: str, retrieval: str) -> Config:
    """Return the P3 configuration of these settings."""
    return Config(f"exp2_P3_{generator}_{dictionary}_{vocab.replace('+', '_')}_{retrieval}",
                  "exp2", generator=generator, dictionary=dictionary, placement="P3",
                  vocab=vocab, retrieval=retrieval)


def link_note(config_id: str) -> str:
    """Describe the fold-1 links of a D2 configuration (count, non-exact share)."""
    spec, theta = evaluations[config_id]["spec"], evaluations[config_id]["fold_theta"][1]
    p3 = spec["placement"] == "P3"
    linked = links(dictionaries["D2"][1], spec["generator"] if p3 else "", 1,
                   theta["tau_link"] if p3 else None)
    total = sum(len(members) for members in linked.members.values())
    return (f"fold 1 links: {total}, non-exact {len(linked.non_exact)} "
            f"({len(linked.non_exact) / max(1, total):.0%})")


exp2_ids = []
for _dictionary in DICTIONARY_NAMES:
    exp2_ids.append(evaluate(Config(f"exp2_P1_{_dictionary}", "exp2", dictionary=_dictionary,
                                    placement="P1")))
    exp2_ids += [evaluate(Config(f"exp2_P2_{g}_{_dictionary}", "exp2", generator=g,
                                 dictionary=_dictionary, placement="P2"))
                 for g in CARRIED_GENERATORS]
_settings = [p3_config(CARRIED_GENERATORS[0], "D3", vocab, retrieval)
             for vocab in ("archive", "archive+dict") for retrieval in ("cutoff", "top5")]
exp2_ids += [evaluate(config) for config in _settings]
best_p3_setting = max(_settings, key=lambda c: evaluations[c.config_id]["primary"])
for _generator in CARRIED_GENERATORS:
    for _dictionary in DICTIONARY_NAMES:
        _config = p3_config(_generator, _dictionary, best_p3_setting.vocab,
                            best_p3_setting.retrieval)
        if _config.config_id not in evaluations:
            exp2_ids.append(evaluate(_config))
carried_exp2 = sorted(exp2_ids, key=lambda c: -evaluations[c]["primary"])[:2]
for _config_id in exp2_ids:
    if evaluations[_config_id]["spec"]["dictionary"] == "D2":
        evaluations[_config_id]["notes"] = link_note(_config_id)

_best_p3 = max((c for c in exp2_ids if "_P3_" in c), key=lambda c: evaluations[c]["primary"])
_spec = evaluations[_best_p3]["spec"]
_links = links(dictionaries[_spec["dictionary"]][1], _spec["generator"], 1,
               evaluations[_best_p3]["fold_theta"][1]["tau_link"])
_sample = sorted(_links.non_exact.items())
np.random.default_rng(SEED).shuffle(_sample)
write_csv(REVIEW_DIR / "link_sample.csv", [
    {"config_id": _best_p3, "fold": 1, "archive_form": forms[form][2],
     "linked_through": forms[through][2], "cosine": round(cosine, 3),
     "groups": " ".join(sorted(_links.groups_of_form[form])), "reviewer_ok": ""}
    for form, (through, cosine) in _sample[:100]])
report("Experiment 2: dictionaries × placements (fold-mean macro R@10, pooled parameters)",
       exp2_ids)
print(f"{RULE}\n  best P3 setting: {best_p3_setting.vocab}, {best_p3_setting.retrieval}; "
      f"carried forward: {', '.join(carried_exp2)}")

────────────────────────────────────────────────────────────────────────
Experiment 2: dictionaries × placements (fold-mean macro R@10, pooled parameters)
────────────────────────────────────────────────────────────────────────
  exp2_P3_E4_D3_archive_dict_cutoff    R@10 0.885   (c 0.7, tau_link 0.9, lam 0.9, τ_d 0.78)
  exp2_P3_E1_D3_archive_dict_cutoff    R@10 0.876   (c 0.7, tau_link 0.85, lam 0.9, τ_d 0.70)
  exp2_P3_E1_D3_archive_cutoff         R@10 0.873   (c 0.7, tau_link 0.85, lam 0.9, τ_d 0.68)
  exp2_P3_E4_D1_archive_dict_cutoff    R@10 0.868   (c 0.7, tau_link 0.9, lam 0.9, τ_d 0.73)
  exp2_P3_E4_D2_archive_dict_cutoff    R@10 0.865   (c 0.7, tau_link 0.9, lam 0.9, τ_d 0.75)
  exp2_P2_E4_D3                        R@10 0.855   (c 0.7, lam 0.9, τ_d 0.70)
  exp2_P3_E1_D2_archive_dict_cutoff    R@10 0.851   (c 0.7, tau_link 0.8, lam 0.9, τ_d 0.50)
  exp2_P2_E1_D3                        R@10 0.845   (c 0.7, lam 0.9, τ_d 0.50)
  exp2_P3_E1_D1_archive_dict_cutoff    R@10 0.845   (c

**7. Inputs of Experiment 3**

This cell prepares everything Experiment 3 needs, so that `step_6_rank_and_freeze.ipynb` can rank and fuse without the generators and dictionaries. Experiment 3 pools the candidates of Experiment 2's two carried configurations and of the best lexical generator, each at its own fold's tuned parameters. For every query, it records:

1. each of the three components' direct and dictionary-derived similarities to every index token, and the time each took, with the index tokens' persons and fields in the column order of these arrays
2. the raw signals the learned scorer K4 takes as features: each component embedding's cosine without a cut-off, the trigram, Daitch–Mokotoff and Beider–Morse matches, and the initial-sound rule D4, with the time they took

It writes `./cache/exp3_inputs.npz` and `./cache/development_runs.json`, the record of every configuration of both experiments and of the configurations carried forward.

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# Inputs of Experiment 3  —  token similarities of the pooled components
#
# Every query contributes one row per query token to each array, so the
# arrays share one row index; `row_start` gives each query's first row.
# Masked tokens are already zero, as they were when the components ran.
# ─────────────────────────────────────────────────────────────────────────────
if "carried_exp2" not in globals():
    raise RuntimeError("Run the 'Experiment 2' cell first.")

COMPONENTS = [*carried_exp2, best_lexical]
K4_SIGNALS = [*sorted({evaluations[c]["spec"]["generator"] for c in COMPONENTS
                       if evaluations[c]["spec"]["generator"] in EMBEDDINGS}),
              "L2", "L3", "L4", "D4"]


def component_config(config_id: str) -> Config:
    """Rebuild the `Config` of an evaluated configuration."""
    spec = evaluations[config_id]["spec"]
    return Config(**{k: v for k, v in spec.items() if k not in ("ranker", "aggregation")})


def raw_signal(generator: str, query: Query) -> np.ndarray:
    """Return a signal's similarity of each query token to every index token
    (no cut-off, no cap), with the query's masked tokens set to zero."""
    rows = np.array([generator_similarity(generator, query.fold, t)[tok_form]
                     for t in query.tokens])
    rows[:, query.masked] = 0
    return rows


exp3_arrays = defaultdict(list)
exp3_ms = np.zeros((len(queries), len(COMPONENTS) + 1))
for _i, _query in enumerate(queries):
    for _k, _config_id in enumerate(COMPONENTS):
        _started = time.perf_counter()
        _direct, _derived = component_similarity(
            component_config(_config_id), evaluations[_config_id]["fold_theta"][_query.fold],
            _query)
        exp3_ms[_i, _k] = (time.perf_counter() - _started) * 1000
        exp3_arrays[f"direct_{_k}"].append(_direct.astype(np.float32))
        exp3_arrays[f"derived_{_k}"].append(_derived.astype(np.float32))
    _started = time.perf_counter()
    for _signal in K4_SIGNALS:
        exp3_arrays[f"signal_{_signal}"].append(raw_signal(_signal, _query).astype(np.float32))
    exp3_ms[_i, -1] = (time.perf_counter() - _started) * 1000
np.savez_compressed(Path("cache/exp3_inputs.npz"),
                    query_ids=np.array([q.query_id for q in queries]),
                    person_ids=np.array(person_ids), tok_person=tok_person,
                    tok_surname=tok_surname,
                    row_start=np.cumsum([0] + [len(q.tokens) for q in queries]),
                    ms=exp3_ms, **{k: np.concatenate(v) for k, v in exp3_arrays.items()})
development_runs = {
    "configurations": evaluations, "not_run": not_run_configs,
    "carried": {"exp1_embeddings": carried_embeddings, "exp1_lexical": best_lexical,
                "exp2": carried_exp2, "exp2_best_p3_setting": asdict(best_p3_setting)},
    "exp3": {"components": COMPONENTS, "k4_signals": K4_SIGNALS},
}
Path("cache/development_runs.json").write_text(json.dumps(development_runs, indent=1) + "\n",
                                                encoding="utf-8")

print(f"{RULE}\nInputs of Experiment 3: {len(queries)} queries\n{RULE}")
for _k, _config_id in enumerate(COMPONENTS):
    print(f"  component {_k}  {_config_id:<36} p50 {np.percentile(exp3_ms[:, _k], 50):6.2f} ms")
print(f"  K4 signals   {', '.join(K4_SIGNALS):<36} p50 {np.percentile(exp3_ms[:, -1], 50):6.2f} ms")
print(f"{RULE}\n  {sum(len(q.tokens) for q in queries)} query-token rows × {len(tok_form)} "
      f"index tokens written for {len(exp3_arrays)} arrays")

────────────────────────────────────────────────────────────────────────
Inputs of Experiment 3: 1743 queries
────────────────────────────────────────────────────────────────────────
  component 0  exp2_P3_E4_D3_archive_dict_cutoff    p50   0.74 ms
  component 1  exp2_P3_E1_D3_archive_dict_cutoff    p50   0.70 ms
  component 2  exp1_L3                              p50   0.03 ms
  K4 signals   E1, E4, L2, L3, L4, D4               p50   0.24 ms
────────────────────────────────────────────────────────────────────────
  3445 query-token rows × 1844 index tokens written for 12 arrays


**8. Consistency Check**

This cell reads `./cache/development_runs.json`, every run file and `./cache/exp3_inputs.npz` back from disk and checks them independently of the cells above:

1. every configuration the manifest's §6 requires for Experiments 1 and 2 is recorded as evaluated or not run, and every evaluated one has its run file, aligned query for query with `queries_dev.csv`
2. each configuration's primary score, recomputed from its run file with plain loops (the relevant persons among the displayed first ten, per query, averaged per query set, per fold and over folds), equals the recorded one
3. no run names a person outside the development split, and the Experiment 3 arrays hold one row per query token

It ends with the configurations checked and the result of the check.

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# Consistency check  —  the runs against the development queries
#
# The R@10 of each query is recounted from the stored ids, scores and
# display cut-off, without the vectorised metric code of the engine.
# ─────────────────────────────────────────────────────────────────────────────
if "development_runs" not in globals():
    raise RuntimeError("Run every cell above first.")


def required_ids(stored: dict) -> set[str]:
    """Return the configuration ids §6 requires for Experiments 1 and 2."""
    carried = [stored["configurations"][c]["spec"]["generator"]
               for c in stored["carried"]["exp1_embeddings"]]
    setting = stored["carried"]["exp2_best_p3_setting"]
    required = {f"exp1_{g}" for g in (*EMBEDDINGS, *LEXICAL)}
    required |= {f"exp2_P1_{d}" for d in DICTIONARY_NAMES}
    required |= {f"exp2_P2_{g}_{d}" for g in carried for d in DICTIONARY_NAMES}
    return required | {p3_config(g, d, setting["vocab"], setting["retrieval"]).config_id
                       for g in carried for d in DICTIONARY_NAMES}


def recounted_primary(config_id: str) -> float:
    """Recount a configuration's fold-mean macro R@10 from its run file."""
    stored = np.load(RUNS_DIR / f"{config_id}.npz")
    by_fold = defaultdict(lambda: defaultdict(list))
    for i, query in enumerate(queries):
        shown = [p for p, s in zip(stored["ids"][i], stored["scores"][i])
                 if p >= 0 and s > 0 and s >= stored["tau"][i]][:10]
        found = sum(p in query.relevant for p in shown)
        by_fold[query.fold][query.query_set].append(found / min(10, len(query.relevant)))
    return float(np.mean([np.mean([np.mean(v) for v in sets.values()])
                          for sets in by_fold.values()]))


def run_problems(stored: dict) -> list[str]:
    """Check every evaluated configuration's run file and primary score."""
    problems = []
    query_ids = [q.query_id for q in queries]
    for config_id, record in stored["configurations"].items():
        path = RUNS_DIR / f"{config_id}.npz"
        if not path.exists() or list(np.load(path)["query_ids"]) != query_ids:
            problems.append(f"{config_id}: run file missing or misaligned")
        elif abs(recounted_primary(config_id) - record["primary"]) > 1e-9:
            problems.append(f"{config_id}: primary {record['primary']:.4f} differs when recounted")
        elif np.load(path)["ids"].max() >= len(persons):
            problems.append(f"{config_id}: names a person outside the development split")
    return problems


_stored = json.loads(Path("cache/development_runs.json").read_text(encoding="utf-8"))
_inputs = np.load(Path("cache/exp3_inputs.npz"))
_checks = {
    "required configurations": [f"{c}: not recorded" for c in required_ids(_stored)
                                if c not in {**_stored["configurations"], **_stored["not_run"]}],
    "run files and primary scores": run_problems(_stored),
    "Experiment 3 inputs": [f"{name}: {_inputs[name].shape[0]} rows" for name in _inputs.files
                            if name.startswith(("direct", "derived", "signal"))
                            and _inputs[name].shape != (_inputs["row_start"][-1], len(tok_form))],
}
print(f"{RULE}\nConsistency: cache/ against the development queries\n{RULE}")
for _name, _problems in _checks.items():
    print(f"  {'!! ' if _problems else 'OK '}{_name:<30} "
          + (f"{len(_problems)} problem(s)" if _problems else "consistent"))
    for _problem in _problems[:10]:
        print(f"        - {_problem}")
_consistent = not any(_checks.values())
print(f"{RULE}\n  {len(_stored['configurations'])} configurations evaluated, "
      f"{len(_stored['not_run'])} not run  ({'consistent' if _consistent else 'inconsistent'})")
print("\n✓ Experiments 1 and 2 complete (runs, carried configurations, Experiment 3 inputs).")
print(f"  development runs consistent: {_consistent}")

────────────────────────────────────────────────────────────────────────
Consistency: cache/ against the development queries
────────────────────────────────────────────────────────────────────────
  OK required configurations        consistent
  OK run files and primary scores   consistent
  OK Experiment 3 inputs            consistent
────────────────────────────────────────────────────────────────────────
  27 configurations evaluated, 0 not run  (consistent)

✓ Experiments 1 and 2 complete (runs, carried configurations, Experiment 3 inputs).
  development runs consistent: True
